### Loading 

In [ ]:
from pathlib import Path
import pandas as pd

from momics.utils import load_and_clean
import mgnify_methods.paper_modules as pm
from mgnify_methods.taxonomy import (
    # fill_lower_taxa,
    pivot_taxonomic_data,
    aggregate_by_taxonomic_level,
)

from mgnify_methods.taxonomy import (
    remove_singletons_per_sample,
    prevalence_cutoff_abund,
)

from mgnify_methods.metacomp.transforms import (
    apply_transform_method,
)

In [ ]:
root_dir = Path().resolve().parent
root_dir

WindowsPath('C:/Users/David Palecek/Documents/Python_projects/emobon/emobon-basic-models')

In [ ]:
contig_path = root_dir / "configs" / "model_test.json"
CONFIG = pm.config_setup(root_dir, contig_path)

INFO | mgnify_methods.paper_modules | Directory ready: C:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\outputs\analysis_20260304_1737
INFO | mgnify_methods.paper_modules | Directory ready: C:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\notebooks\analysis_cache
INFO | mgnify_methods.paper_modules | Configuration saved to: C:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\outputs\analysis_20260304_1737\analysis_config.json
INFO | mgnify_methods.paper_modules | Logger configured to write to C:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\outputs\analysis_20260304_1737\analysis.log


In [ ]:
# loads both ssu and emobon metadata, 
abundance_emobon, emobon_meta = pm.load_emobon(root_dir, ret='ssu')
abundance_emobon.shape, emobon_meta.shape

c:\Users\David Palecek\Documents\Python_projects\emobon\emobon-basic-models\.venv\Lib\site-packages\momics\metadata.py:263: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata["replicate_info"] = (


((111320, 9), (181, 100))

In [ ]:
abundance_emobon = pm.process_emobon_data(abundance_emobon, CONFIG)
abundance_emobon.shape

In [ ]:
# turn abundance into taxonomy for cleaning lineaages
taxonomy_table = pm.clean_abundance_table(abundance_emobon, CONFIG)

tax_level = CONFIG['taxonomy']['analysis_level']
abundance_table = aggregate_by_taxonomic_level(
    taxonomy_table.df_filt,
    level=tax_level,
    dropna=CONFIG['diversity']['alpha']['dropna']
)
del taxonomy_table

INFO | mgnify_methods.tables.tables | Converting source: tax_mgnify_raw
WARNING | mgnify_methods.tables.tables | DataFrame is abundance without ncbi_tax_id data.
INFO | mgnify_methods.tables.tables | self.source before conversion to Taxonomy table: tax_mgnify_raw
INFO | mgnify_methods.tables.tables | Converting source: tax_processed
INFO | mgnify_methods.tables.sources.converters | Abundance dtype after conversion: uint16
INFO | mgnify_methods.tables.tables | Validating source: tax_processed
INFO | mgnify_methods.paper_modules | Taxonomy table ready: (111320, 9)


NameError: name 'aggregate_by_taxonomic_level' is not defined

In [ ]:
abundance_table = pivot_taxonomic_data(abundance_table)
type(abundance_table)

## Processing of the raw abundance table
- remove singletons
- CLR

In [ ]:
abundance_table_beta = remove_singletons_per_sample(abundance_table, skip_columns=0)

abundance_table_beta = prevalence_cutoff_abund(
        abundance_table, 
        percent=CONFIG['preprocessing']['prevalence_cutoff'],
        skip_columns=0
    )

preprocess_tables = {'emobon': abundance_table}

## Data transformation for beta diversity
if CONFIG['transform']['enabled']:
    transformed_tables = {}
    for sample_type, table_or_dict in preprocess_tables.items():
        if isinstance(table_or_dict, dict):
            transformed_tables[sample_type] = {
                tax_level: apply_transform_method(df, CONFIG)
                for tax_level, df in table_or_dict.items()
            }
        else:
            transformed_tables[sample_type] = apply_transform_method(table_or_dict, CONFIG)

    preprocess_tables = transformed_tables

## Modelling